# Automate GROMACS analysis 

In [1]:
import glob
import os
import subprocess

In [3]:
root_data_dir = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/data/raw'
notebook_dir = '/lustre/ea-nrtmidas/users/3770/emulsion_stability/notebooks'

In [ ]:
'''
identify locations where the .xtc and .gro files are present and accordingly create bash scripts to be executed using subprocess
analysis for rdf distribution
'''
diff_systems = glob.glob(f'{root_data_dir}/self_assembly_polymer_surfactant*_toluene_water_*')
diff_systems = [item for item in diff_systems if 'surfactant240' not in item] # ignoring systems where simulations are in progress

for system in diff_systems:
    stages = glob.glob(f'{system}/equilibration_*') + glob.glob(f'{system}/production_*')
    for stage in stages:
        for config in glob.glob(f'{stage}/*'):
            xtc_files = glob.glob(f'{config}/*.xtc')
            gro_files = glob.glob(f'{config}/*.gro')
            gro_files = [item for item in gro_files if item.split('/')[-3].split('_')[-1].lower() in item.split('/')[-1].lower()] # ensuring the copied files to each directory is not counted multiple times
            
            # rdf analysis frequency checker if using .xtc file
            if 'production' in stage:
                freq = 500000
            elif 'equilibration_NVT' in stage:
                freq = 25000
            elif 'equilibration_NPT' in stage:
                freq = 50000
            
            # check if analysis files are already run for this directory
            thermo_files = glob.glob(f'{config}/thermo/*.xvg')
            if(0==1):
                continue
            else:
                print(thermo_files)
                for xtc_file in xtc_files:
                    gro_file = xtc_file[:-4]+'.gro'
                    tpr_file = xtc_file[:-4]+'.tpr'
                    edr_file = xtc_file[:-4]+'.edr'
                    print(xtc_file)
                    bash_script = f"""
                    echo "Starting automation..."
                    vpkg_require gromacs

                    # Create necessary index file for grouping atoms
                    echo -e '"PS8B" & a B5 | a B6 | a B7 | a B8 | a B9 | a B11 | a B12 | a B13 | a B14 | a B15\nname 6 Head_3_4\n"PS8B"  & a B19 | a B20 | a B21 | a B22 | a B24 | a B25 | a B26 | a B27\nname 7 Head_1_2\n"PS8B" & a C32 | a C33 | a C0 | a C1\nname 8 Tail\n"LMA" & a MB\nname 9 Poly_MB\n"LMA" & a ME\nname 10 Poly_ME\n"LMA" & a MT\nname 11 Poly_MT\n"LMA" & a E1\nname 12 Poly_E1\n"LMA" & a E2\nname 13 Poly_E2\nq' | mpirun gmx make_ndx -f {gro_file} -o {config}/custom.ndx

                    # rdf analysis
                    mkdir {config}/rdf_analysis
                    mpirun gmx rdf -f {gro_file} -s {tpr_file} -n {config}/custom.ndx -ref Tail -sel Tail Head_1_2 Head_3_4 Poly_MB Poly_ME Poly_MT Poly_E1 Poly_E2 W Tolue -o {config}/rdf_analysis/rdf_ref_Tail.xvg -bin 0.05
                    mpirun gmx rdf -f {gro_file} -s {tpr_file} -n {config}/custom.ndx -ref Head_1_2 -sel Head_1_2 Tail Head_3_4 Poly_MB Poly_ME Poly_MT Poly_E1 Poly_E2 W Tolue -o {config}/rdf_analysis/rdf_ref_Head12.xvg -bin 0.05
                    mpirun gmx rdf -f {gro_file} -s {tpr_file} -n {config}/custom.ndx -ref Head_3_4 -sel Head_3_4 Tail Head_1_2 Poly_MB Poly_ME Poly_MT Poly_E1 Poly_E2 W Tolue -o {config}/rdf_analysis/rdf_Head34.xvg -bin 0.05
                    mpirun gmx rdf -f {gro_file} -s {tpr_file} -n {config}/custom.ndx -ref W -sel Tail Head_1_2  Head_3_4 Poly_MB Poly_ME Poly_MT Poly_E1 Poly_E2 Tolue -o {config}/rdf_analysis/rdf_ref_W.xvg -bin 0.05
                    mpirun gmx rdf -f {gro_file} -s {tpr_file} -n {config}/custom.ndx -ref Tolue -sel Tolue Tail Head_1_2  Head_3_4 Poly_MB Poly_ME Poly_MT Poly_E1 Poly_E2 W -o {config}/rdf_analysis/rdf_ref_Tolue.xvg -bin 0.05
                    mpirun gmx rdf -f {gro_file} -s {tpr_file} -n {config}/custom.ndx -ref Poly_MB -sel Poly_MB Tolue Tail Head_1_2  Head_3_4 Poly_ME Poly_MT Poly_E1 Poly_E2 W -o {config}/rdf_analysis/rdf_ref_MB.xvg -bin 0.05
                    mpirun gmx rdf -f {gro_file} -s {tpr_file} -n {config}/custom.ndx -ref Poly_E2 -sel Poly_MB Tolue Tail Head_1_2  Head_3_4 Poly_ME Poly_MT Poly_E1 Poly_E2 W -o {config}/rdf_analysis/rdf_ref_E2.xvg -bin 0.05

                    # thermo properties
                    mkdir {config}/thermo
                    echo -e "Temperature\nPressure\nDensity\nTotal-Energy\nPotential\nKinetic-En.\nBox-Vol\n0" | mpirun gmx energy -f {edr_file} -o {config}/thermo/thermo.xvg

                    echo "Done."
                    """
                    subprocess.run(bash_script, shell=True, executable='/bin/bash')


To identify the number of clusters, my initial approach was to only consider the polymer beads and since I know aprior the polymer beads will be the core of the micelle, tracking the clusters of polymers indirectly will then result to tracking micelle counts. I initially preferred this approach, as I donot have to consider the effect of periodic boundary condition.

In [ ]:
'''
identify locations where the .xtc and .gro files are present and accordingly create bash scripts to be executed using subprocess
analysis for number of clusters
'''
diff_systems = glob.glob(f'{root_data_dir}/self_assembly_polymer_surfactant*_toluene_water_*')
diff_systems = [item for item in diff_systems if 'surfactant240' not in item] # ignoring systems where simulations are in progress

for system in diff_systems:
    stages = glob.glob(f'{system}/equilibration_*') + glob.glob(f'{system}/production_*')
    for stage in stages:
        for config in glob.glob(f'{stage}/*'):
            xtc_files = glob.glob(f'{config}/*.xtc')
            gro_files = glob.glob(f'{config}/*.gro')
            gro_files = [item for item in gro_files if item.split('/')[-3].split('_')[-1].lower() in item.split('/')[-1].lower()] # ensuring the copied files to each directory is not counted multiple times
            
            # check if analysis files are already run for this directory
            cluster_files = glob.glob(f'{config}/cluster/my_num_clusters.xvg')
            if(len(cluster_files)!=0):
                continue
            else:
                for xtc_file in xtc_files:
                    if('whole.xtc' in xtc_file):
                        continue
                    gro_file = xtc_file[:-4]+'.gro'
                    tpr_file = xtc_file[:-4]+'.tpr'
                    edr_file = xtc_file[:-4]+'.edr'
                    print(xtc_file)
                    bash_script = f"""
                    echo "Starting automation..."
                    vpkg_require gromacs

                    # Create necessary index file for grouping atoms
                    echo -e '"PS8B" & a B5 | a B6 | a B7 | a B8 | a B9 | a B11 | a B12 | a B13 | a B14 | a B15\nname 6 Head_3_4\n"PS8B"  & a B19 | a B20 | a B21 | a B22 | a B24 | a B25 | a B26 | a B27\nname 7 Head_1_2\n"PS8B" & a C32 | a C33 | a C0 | a C1\nname 8 Tail\n"LMA" & a MB\nname 9 Poly_MB\n"LMA" & a ME\nname 10 Poly_ME\n"LMA" & a MT\nname 11 Poly_MT\n"LMA" & a E1\nname 12 Poly_E1\n"LMA" & a E2\nname 13 Poly_E2\nq' | mpirun gmx make_ndx -f {gro_file} -o {config}/custom.ndx
                    
                    # make molecules whole
                    # echo -e '0' | mpirun gmx trjconv -f {xtc_file} -s {tpr_file} -pbc whole -o {config}/whole.xtc

                    # thermo properties
                    rm -r {config}/cluster {config}/cluster_whole # removing any residual directories
                    mkdir -p {config}/cluster
                    mkdir -p {config}/cluster_whole # to check if there is any difference than the actual calculation

                    echo -e '3' | mpirun gmx clustsize -f {xtc_file} -s {tpr_file} -nc {config}/cluster/my_num_clusters.xvg -mc {config}/cluster/my_max_size.xvg -n {config}/custom.ndx -cut 2.2
                    # echo -e '3' | mpirun gmx clustsize -f {config}/whole.xtc -s {tpr_file} -nc {config}/cluster_whole/my_num_clusters.xvg -mc {config}/cluster_whole/my_max_size.xvg -n {config}/custom.ndx

                    echo 'Done.'
                    """
                    subprocess.run(bash_script, shell=True, executable='/bin/bash')


Since I am also interested in the distribution of micelle, the previous trick of only tracking polymer thus not work. So, I resort to tracking the entire micelle. To my knowledge, I donot have to worry about the periodic boundary condition, since `clustsize` in itself will take care of it.

In [ ]:
'''
identify locations where the .xtc and .gro files are present and accordingly create bash scripts to be executed using subprocess
analysis for number of clusters
'''
diff_systems = glob.glob(f'{root_data_dir}/self_assembly_polymer_surfactant*_toluene_water_*')
diff_systems = [item for item in diff_systems if 'surfactant240' not in item] # ignoring systems where simulations are in progress

for system in diff_systems:
    stages = glob.glob(f'{system}/equilibration_*') + glob.glob(f'{system}/production_*')
    for stage in stages:
        for config in glob.glob(f'{stage}/*'):
            xtc_files = glob.glob(f'{config}/*.xtc')
            gro_files = glob.glob(f'{config}/*.gro')
            gro_files = [item for item in gro_files if item.split('/')[-3].split('_')[-1].lower() in item.split('/')[-1].lower()] # ensuring the copied files to each directory is not counted multiple times
            
            # check if analysis files are already run for this directory
            cluster_files = glob.glob(f'{config}/cluster/my_num_clusters.xvg')
            if(len(cluster_files)!=0):
                continue
            else:
                for xtc_file in xtc_files:
                    if('whole.xtc' in xtc_file):
                        continue
                    gro_file = xtc_file[:-4]+'.gro'
                    tpr_file = xtc_file[:-4]+'.tpr'
                    edr_file = xtc_file[:-4]+'.edr'
                    print(xtc_file)
                    bash_script = f"""
                    echo "Starting automation..."
                    vpkg_require gromacs

                    # Create necessary index file for grouping atoms into micelle
                    echo -e '2 | 3 | 4\nname 6 micelle\nq' | mpirun gmx make_ndx -f {gro_file} -o {config}/micelle.ndx

                    # thermo properties
                    mkdir -p {config}/micelle_cluster

                    echo -e '6' | mpirun gmx clustsize -f {xtc_file} -s {tpr_file} -nc {config}/micelle_cluster/my_num_clusters.xvg -mc {config}/micelle_cluster/my_max_size.xvg -n {config}/micelle.ndx -cut 2.2

                    echo 'Done.'
                    """
                    subprocess.run(bash_script, shell=True, executable='/bin/bash')
